# Command Execution Report
This notebook runs the requested shell commands in `/workspaces/onemonetry`, captures exact `stdout`, `stderr`, and exit codes, and continues even when commands fail.

## 1. Set Working Directory and Command Runner
Creates a helper that runs shell commands from `/workspaces/onemonetry`, captures exact output streams, captures exit codes, and never aborts on failure.

In [ ]:
from __future__ import annotations

import json
import subprocess
from pathlib import Path

WORKDIR = "/workspaces/onemonetry"
results = []


def run_command(index: int, command: str):
    proc = subprocess.run(
        command,
        shell=True,
        cwd=WORKDIR,
        text=True,
        capture_output=True,
        executable="/bin/bash",
    )
    result = {
        "index": index,
        "command": command,
        "cwd": WORKDIR,
        "stdout": proc.stdout,
        "stderr": proc.stderr,
        "exit_code": proc.returncode,
        "status": "succeeded" if proc.returncode == 0 else "failed",
    }
    results.append(result)
    return result


print(f"Working directory: {WORKDIR}")
print("Runner ready.")

## 2. Execute Command 1 (dry_run=true Purge Placeholders)
## 3. Execute Command 2 (dry_run=false Purge Placeholders)
## 4. Execute Command 3 (Level 5 Smoke Test)
## 5. Execute Command 4 (Read /tmp/level5-generate.json)
## 6. Execute Command 5 (Read /tmp/level5-sequence-latest.json)
## 7. Execute Command 6 (Detect Placeholder Tokens)
## 8. Execute Command 7 (Detect Signature Lines)
Runs each requested command, captures exact stdout/stderr and exit codes, and continues regardless of failures.

In [ ]:
cmd1 = "curl -sS -X POST http://localhost:8000/api/email/sequences/ch-09523048/purge-placeholders -H 'Content-Type: application/json' -d '{\"dry_run\":true}'"
cmd2 = "curl -sS -X POST http://localhost:8000/api/email/sequences/ch-09523048/purge-placeholders -H 'Content-Type: application/json' -d '{\"dry_run\":false}'"
cmd3 = "SKIP_LOGIN=1 bash scripts/level5-smoke.sh ch-09523048"
cmd4 = "cat /tmp/level5-generate.json"
cmd5 = "cat /tmp/level5-sequence-latest.json"
cmd6 = r"grep -Eo '\\[(rounded figure|Your Name|Your Title|AE_NAME|AE_TITLE)\\]' /tmp/level5-sequence-latest.json || echo no placeholders"
cmd7 = r"grep -E 'Account Executive \\| Revolut Business|revolut\\.com/business|(^|[[:space:]])(Best|Regards|Thanks),?$' /tmp/level5-sequence-latest.json || echo no signature lines"

r1 = run_command(1, cmd1)
r2 = run_command(2, cmd2)
r3 = run_command(3, cmd3)
r4 = run_command(4, cmd4)
r5 = run_command(5, cmd5)
r6 = run_command(6, cmd6)
r7 = run_command(7, cmd7)

for r in [r1, r2, r3, r4, r5, r6, r7]:
    print(f"command {r['index']}: exit_code={r['exit_code']} ({r['status']})")

## 9. Aggregate and Display Exact stdout/stderr and Exit Codes
Renders a per-command report with command, working directory, stdout, stderr, exit code, and success/failure status.

In [ ]:
for r in results:
    print(f"=== COMMAND {r['index']} ===")
    print(f"COMMAND: {r['command']}")
    print(f"CWD: {r['cwd']}")
    print(f"EXIT_CODE: {r['exit_code']}")
    print(f"STATUS: {r['status']}")
    print("STDOUT_BEGIN")
    print(r["stdout"], end="")
    if not r["stdout"].endswith("\n"):
        print()
    print("STDOUT_END")
    print("STDERR_BEGIN")
    print(r["stderr"], end="")
    if not r["stderr"].endswith("\n"):
        print()
    print("STDERR_END")

## 10. Persist Execution Report to JSON/Markdown Artifact
Writes the full machine-readable JSON report and a human-readable Markdown summary to workspace files for later inspection.

In [ ]:
json_path = Path(WORKDIR) / "tmp-command-results.json"
md_path = Path(WORKDIR) / "tmp-command-results.md"

json_path.write_text(json.dumps(results, indent=2), encoding="utf-8")

lines = ["# Command Results", ""]
for r in results:
    lines.append(f"## Command {r['index']}")
    lines.append(f"- Command: `{r['command']}`")
    lines.append(f"- CWD: `{r['cwd']}`")
    lines.append(f"- Exit code: `{r['exit_code']}`")
    lines.append(f"- Status: `{r['status']}`")
    lines.append("- Stdout:")
    lines.append("```text")
    lines.append(r["stdout"])
    lines.append("```")
    lines.append("- Stderr:")
    lines.append("```text")
    lines.append(r["stderr"])
    lines.append("```")
    lines.append("")

md_path.write_text("\n".join(lines), encoding="utf-8")

print(str(json_path))
print(str(md_path))